In [1]:
import io
import pandas as pd
import  numpy as np

# URL do arquivo JSON contendo os dados de hospedagem
url = 'https://raw.githubusercontent.com/YuriArduino/Estudos_Pandas/refs/heads/data-tests/dados_hospedagem.json'

# Lê o arquivo JSON da URL
# Utiliza io.StringIO para tratar a URL como um arquivo em memória
# Converte o JSON lido para DataFrame, depois para JSON string
# e lê novamente para garantir o formato correto
dados = pd.read_json(io.StringIO(pd.read_json(url).to_json()))
dados = pd.json_normalize(dados['info_moveis']) #SELECIONAR A COLUNA A NORMALIZAR

colunas = list(dados.columns)
# Explode as colunas a partir da quarta para lidar com listas/estruturas aninhadas
dados = dados.explode(colunas[3:])
dados.reset_index(inplace=True, drop=True) # Reseta o índice e remove o índice antigo

# Converte colunas numéricas para tipo inteiro (int64)
col_numericas = ['max_hospedes','quantidade_banheiros','quantidade_quartos','quantidade_camas']
dados[col_numericas] = dados[col_numericas].astype(np.int64)

# Converte a coluna de avaliação geral para tipo float (float64)
dados['avaliacao_geral'] = dados['avaliacao_geral'].astype(np.float64)

# Remove '$' e ',' e converte colunas de preço e taxas para float
dados[['preco','taxa_deposito','taxa_limpeza']] = dados[
    ['taxa_deposito','taxa_limpeza','preco']].map(
    lambda x: float(x.replace('$', '').replace(',','').strip()))

# Converte explicitamente para float64 (redundante após o map, mas mantém a clareza)
dados[['preco','taxa_deposito','taxa_limpeza']] = dados[
    ['preco','taxa_deposito','taxa_limpeza']].astype(np.float64)

# Mostra as primeiras 5 linhas do DataFrame
dados.head()

,avaliacao_geral,experiencia_local,max_hospedes,descricao_local,descricao_vizinhanca,quantidade_banheiros,quantidade_quartos,quantidade_camas,modelo_cama,comodidades,taxa_deposito,taxa_limpeza,preco
0,10.0,--,1,This clean and comfortable one bedroom sits ri...,Lower Queen Anne is near the Seattle Center (s...,1,1,1,Real Bed,"{Internet,""Wireless Internet"",Kitchen,""Free Pa...",0.0,110.0,0.0
1,10.0,--,1,Our century old Upper Queen Anne house is loca...,"Upper Queen Anne is a really pleasant, unique ...",1,1,1,Futon,"{TV,Internet,""Wireless Internet"",Kitchen,""Free...",0.0,45.0,0.0
2,10.0,--,1,Cozy room in two-bedroom apartment along the l...,The convenience of being in Seattle but on the...,1,1,1,Futon,"{TV,Internet,""Wireless Internet"",Kitchen,""Free...",0.0,55.0,0.0
3,10.0,--,1,Very lovely and cozy room for one. Convenientl...,"Ballard is lovely, vibrant and one of the most...",1,1,1,Pull-out Sofa,"{Internet,""Wireless Internet"",Kitchen,""Free Pa...",20.0,52.0,0.0
4,10.0,--,1,The “Studio at Mibbett Hollow' is in a Beautif...,--,1,1,1,Real Bed,"{""Wireless Internet"",Kitchen,""Free Parking on ...",15.0,85.0,0.0


#Tokenização

A tokenização é a divisão do texto em tokens, ou seja, unidades menores. Esses tokens podem ser palavras, sílabas, conjuntos de palavras e assim por diante, dependendo do tipo de aplicação.

##Manipulação dos textos

Podemos começar a estruturar a tokenização padronizando o nosso texto para letras minúsculas.

In [2]:
dados['descricao_local']

,descricao_local
0,This clean and comfortable one bedroom sits ri...
1,Our century old Upper Queen Anne house is loca...
2,Cozy room in two-bedroom apartment along the l...
3,Very lovely and cozy room for one. Convenientl...
4,The “Studio at Mibbett Hollow' is in a Beautif...
...,...
3813,Beautiful craftsman home in the historic Wedgw...
3814,Located in a very easily accessible area of Se...
3815,This home is fully furnished and available wee...
3816,This business-themed modern home features: *H...


`str()`

In [3]:
dados['descricao_local'].str.lower() #Transformar para minúscula.

,descricao_local
0,this clean and comfortable one bedroom sits ri...
1,our century old upper queen anne house is loca...
2,cozy room in two-bedroom apartment along the l...
3,very lovely and cozy room for one. convenientl...
4,the “studio at mibbett hollow' is in a beautif...
...,...
3813,beautiful craftsman home in the historic wedgw...
3814,located in a very easily accessible area of se...
3815,this home is fully furnished and available wee...
3816,this business-themed modern home features: *h...


In [4]:
dados['descricao_local'] = dados['descricao_local'].str.lower()
dados.head()

,avaliacao_geral,experiencia_local,max_hospedes,descricao_local,descricao_vizinhanca,quantidade_banheiros,quantidade_quartos,quantidade_camas,modelo_cama,comodidades,taxa_deposito,taxa_limpeza,preco
0,10.0,--,1,this clean and comfortable one bedroom sits ri...,Lower Queen Anne is near the Seattle Center (s...,1,1,1,Real Bed,"{Internet,""Wireless Internet"",Kitchen,""Free Pa...",0.0,110.0,0.0
1,10.0,--,1,our century old upper queen anne house is loca...,"Upper Queen Anne is a really pleasant, unique ...",1,1,1,Futon,"{TV,Internet,""Wireless Internet"",Kitchen,""Free...",0.0,45.0,0.0
2,10.0,--,1,cozy room in two-bedroom apartment along the l...,The convenience of being in Seattle but on the...,1,1,1,Futon,"{TV,Internet,""Wireless Internet"",Kitchen,""Free...",0.0,55.0,0.0
3,10.0,--,1,very lovely and cozy room for one. convenientl...,"Ballard is lovely, vibrant and one of the most...",1,1,1,Pull-out Sofa,"{Internet,""Wireless Internet"",Kitchen,""Free Pa...",20.0,52.0,0.0
4,10.0,--,1,the “studio at mibbett hollow' is in a beautif...,--,1,1,1,Real Bed,"{""Wireless Internet"",Kitchen,""Free Parking on ...",15.0,85.0,0.0


##Regex

Regex é uma expressão regular que se caracteriza por um conjunto de caracteres que fazem uma pesquisa no texto. O Regex é muito utilizado para fazer a busca, validação e manipulação de dados porque, a partir dessa sequência padronizada de caracteres, conseguimos definir os caracteres que queremos ou não selecionar.

In [5]:
#Leitura de texto
dados['descricao_local'][3169]

"built, run and supported by seattle tech and start up veterans, grokhome's focus is to create a supportive environment for smart people working on interesting projects, start ups and more. this listing is an upper bunk, in a 2-person shared room. *note: this fall, there will be major renovations happening on one kitchen and bathroom at a time. there will always be two other working kitchens and two working bathrooms in the house. we'll work to minimize the impact these renovations have on your stay. **this listing is only available to those working in the tech/science space. live in a hacker house, and immerse yourself in the seattle tech scene. you can expect to be surrounded by smart people solving big problems or working on something fun. we have frequent demo nights, and love when our guests share something they are passionate about. if you're new to the city, our deep ties to the seattle tech scene can help you get involved. expand your network, develop your ideas, and learn some

Analisando o texto, podemos notar que existem vários `caracteres especiais` como **vírgulas, pontos, dois pontos e asteriscos**. Esses são os caracteres que queremos remover.

No entanto, precisamos nos atentar que, na língua inglesa, usa-se muito o apóstrofo (') para compor sentidos. Esses sinais não serão removidos.

Além disso, também há o hífen para formar palavras compostas, outro sinal que também não removeremos na nossa substituição. Isso porque palavras compostas possuem um sentido próprio, que difere do significado individual das palavras da composição.


`Método replace()`

Para a substituição desses caracteres especiais, utilizaremos o str junto de outro método de strings, o replace()

###Regex: removendo caracteres especiais


Vamos entender sua estrutura. Nós abrimos e fechamos colchetes e, dentro deles, começamos a expressão com um sinal circunflexo (^), que nega os próximos valores. Ou seja, colocamos em seguida os valores que ele não vai selecionar:

*  a-z: todas as letras, do "a" ao "z", minúsculas;
*  A-Z: todas as letras, do "A" ao "Z", maiúsculas;
*  0-9: todos os números, de 0 a 9;
*  \-: hífen;
*  \': apóstrofo.

Toda essa expressão diz: "selecione todos os caracteres, exceto as letras (maiúsculas ou minúsculas), números, hífen e apóstrofo".

In [6]:
dados['descricao_local'].str.replace(r'[^a-zA-Z0-9\-\']', ' ', regex=True)

,descricao_local
0,this clean and comfortable one bedroom sits ri...
1,our century old upper queen anne house is loca...
2,cozy room in two-bedroom apartment along the l...
3,very lovely and cozy room for one convenientl...
4,the studio at mibbett hollow' is in a beautif...
...,...
3813,beautiful craftsman home in the historic wedgw...
3814,located in a very easily accessible area of se...
3815,this home is fully furnished and available wee...
3816,this business-themed modern home features h...


In [7]:
dados['descricao_local'] = dados['descricao_local'].str.replace(r'[^a-zA-Z0-9\-\']', ' ', regex=True)

###Removendo hifens fora de palavras compostas

O nosso próximo passo será remover os hifens que não estejam conectando palavras compostas, ou seja, os hifens soltos que não tenham nenhum caractere antes ou depois deles

Esse Regex tem uma estrutura diferente: entre parênteses, colocamos um sinal de interrogação (?), um sinal de menor (<), uma exclamação (!), barra () e a letra "w" minúscula. Depois dos primeiros parênteses, inserimos um hífen e abrimos novos parênteses, contendo os mesmos caracteres, exceto pelo sinal de menor. Ou seja:

(?<!\w): uma "lookbehind" negativa que verifica se há um caractere de palavra antes do hífen ou se há apenas um espaço em branco, resultando em "verdadeiro" se não houver nada;
(?!\w): uma lookahead negativa que verifica se há um caractere de palavra depois do hífen.
A combinação de lookbehind negativo e lookahead negativo garante que o hífen esteja entre espaços em branco ou no início ou fim de uma palavra. Nesse caso, esse hífen será selecionado por esse Regex e substituído por um espaço em branco.

In [8]:
dados['descricao_local'].str.replace(r'(?<!\w)-(?!\w)', '', regex=True)

,descricao_local
0,this clean and comfortable one bedroom sits ri...
1,our century old upper queen anne house is loca...
2,cozy room in two-bedroom apartment along the l...
3,very lovely and cozy room for one convenientl...
4,the studio at mibbett hollow' is in a beautif...
...,...
3813,beautiful craftsman home in the historic wedgw...
3814,located in a very easily accessible area of se...
3815,this home is fully furnished and available wee...
3816,this business-themed modern home features h...


In [9]:
dados['descricao_local'] = dados['descricao_local'].str.replace(r'(?<!\w)-(?!\w)', '', regex=True)

In [10]:
dados['descricao_local'][3169]

"built  run and supported by seattle tech and start up veterans  grokhome's focus is to create a supportive environment for smart people working on interesting projects  start ups and more  this listing is an upper bunk  in a 2-person shared room   note  this fall  there will be major renovations happening on one kitchen and bathroom at a time  there will always be two other working kitchens and two working bathrooms in the house  we'll work to minimize the impact these renovations have on your stay    this listing is only available to those working in the tech science space  live in a hacker house  and immerse yourself in the seattle tech scene  you can expect to be surrounded by smart people solving big problems or working on something fun  we have frequent demo nights  and love when our guests share something they are passionate about  if you're new to the city  our deep ties to the seattle tech scene can help you get involved  expand your network  develop your ideas  and learn some

## Para saber mais: aprofundando em regex

Regex (ou expressão regular) é uma sequência de caracteres que define **padrões de busca em texto**. É uma ferramenta poderosa e versátil, permitindo **encontrar, substituir e manipular padrões de forma eficiente**. Regex é amplamente usado em programação, ciência de dados e processamento de texto.

### Aplicações em ciência de dados

* **Limpeza de dados**: localizar e substituir caracteres indesejados em textos brutos.
* **Classificação de texto**: identificar padrões em e-mails, ajudando a classificar mensagens como spam ou não.
* **Extração de informações**: automatizar a análise de grandes volumes de texto e facilitar a organização de dados.

### Recursos e estudo

* Para construir e testar expressões regulares, o site [regex101.com](https://regex101.com/) é muito útil.
* Para aplicações em bancos de dados, o artigo [Principais casos de uso de Regex para tratamento de dados mostra exemplos práticos.](https://cursos.alura.com.br/forum/topico-duvida-duvida-como-posso-melhorar-a-logica-e-visualizacao-520441?hasAccessMGM=true)
---


##Tokenização de strings

`Método split()`

In [11]:
#Transformar uma string numa lista
#Usando o espaço em branco como elemento separador
dados['descricao_local'].str.split()

,descricao_local
0,"[this, clean, and, comfortable, one, bedroom, ..."
1,"[our, century, old, upper, queen, anne, house,..."
2,"[cozy, room, in, two-bedroom, apartment, along..."
3,"[very, lovely, and, cozy, room, for, one, conv..."
4,"[the, studio, at, mibbett, hollow', is, in, a,..."
...,...
3813,"[beautiful, craftsman, home, in, the, historic..."
3814,"[located, in, a, very, easily, accessible, are..."
3815,"[this, home, is, fully, furnished, and, availa..."
3816,"[this, business-themed, modern, home, features..."


In [12]:
dados['descricao_local'] = dados['descricao_local'].str.split()
dados.head()

,avaliacao_geral,experiencia_local,max_hospedes,descricao_local,descricao_vizinhanca,quantidade_banheiros,quantidade_quartos,quantidade_camas,modelo_cama,comodidades,taxa_deposito,taxa_limpeza,preco
0,10.0,--,1,"[this, clean, and, comfortable, one, bedroom, ...",Lower Queen Anne is near the Seattle Center (s...,1,1,1,Real Bed,"{Internet,""Wireless Internet"",Kitchen,""Free Pa...",0.0,110.0,0.0
1,10.0,--,1,"[our, century, old, upper, queen, anne, house,...","Upper Queen Anne is a really pleasant, unique ...",1,1,1,Futon,"{TV,Internet,""Wireless Internet"",Kitchen,""Free...",0.0,45.0,0.0
2,10.0,--,1,"[cozy, room, in, two-bedroom, apartment, along...",The convenience of being in Seattle but on the...,1,1,1,Futon,"{TV,Internet,""Wireless Internet"",Kitchen,""Free...",0.0,55.0,0.0
3,10.0,--,1,"[very, lovely, and, cozy, room, for, one, conv...","Ballard is lovely, vibrant and one of the most...",1,1,1,Pull-out Sofa,"{Internet,""Wireless Internet"",Kitchen,""Free Pa...",20.0,52.0,0.0
4,10.0,--,1,"[the, studio, at, mibbett, hollow', is, in, a,...",--,1,1,1,Real Bed,"{""Wireless Internet"",Kitchen,""Free Parking on ...",15.0,85.0,0.0


###Coluna comodidades

Tokenização mas não está bem estruturada como uma lista.

In [13]:
dados['comodidades']

,comodidades
0,"{Internet,""Wireless Internet"",Kitchen,""Free Pa..."
1,"{TV,Internet,""Wireless Internet"",Kitchen,""Free..."
2,"{TV,Internet,""Wireless Internet"",Kitchen,""Free..."
3,"{Internet,""Wireless Internet"",Kitchen,""Free Pa..."
4,"{""Wireless Internet"",Kitchen,""Free Parking on ..."
...,...
3813,"{TV,""Cable TV"",Internet,""Wireless Internet"",""A..."
3814,"{TV,""Cable TV"",Internet,""Wireless Internet"",Ki..."
3815,"{TV,""Cable TV"",Internet,""Wireless Internet"",""A..."
3816,"{TV,""Cable TV"",Internet,""Wireless Internet"",""A..."


Ao examinar o dataframe, percebemos que as palavras estão entre chaves e separadas por vírgulas. Embora essa estrutura possa se assemelhar a um dicionário, na realidade não se trata da estrutura de um dicionário.

####Tratamento dos dados

Primeiramente, removeremos as chaves e aspas - caracteres que não nos ajudarão a tokenizar nossa coluna de comodidades. Já sabemos fazer isso usando o método replace() e o Regex: vamos identificar os caracteres em questão e substituí-los.

Então, como primeiro parâmetro de replace(), passamos o Regex \{|}|\" para encontrar:

*  {| chave aberta;
*  }| chave fechada;
*  " aspas.

    *    as barras verticais (|) representam uma escolha entre dois padrões diferentes.

Neste caso, podemos manter as vírgulas para utilizá-las como separador para dividir as palavras em uma lista na nossa tokenização.

Além disso, o que vai substituir esses caracteres é uma string vazia. Portanto, como segundo parâmetro, adicionamos apenas um par de aspas simples sem nada entre elas.

>Por fim, passamos regex=True:

In [14]:
dados['comodidades'].str.replace(r'\{|}|\"','',regex=True)

,comodidades
0,"Internet,Wireless Internet,Kitchen,Free Parkin..."
1,"TV,Internet,Wireless Internet,Kitchen,Free Par..."
2,"TV,Internet,Wireless Internet,Kitchen,Free Par..."
3,"Internet,Wireless Internet,Kitchen,Free Parkin..."
4,"Wireless Internet,Kitchen,Free Parking on Prem..."
...,...
3813,"TV,Cable TV,Internet,Wireless Internet,Air Con..."
3814,"TV,Cable TV,Internet,Wireless Internet,Kitchen..."
3815,"TV,Cable TV,Internet,Wireless Internet,Air Con..."
3816,"TV,Cable TV,Internet,Wireless Internet,Air Con..."


In [15]:
dados['comodidades'] = dados['comodidades'].str.replace(r'\{|}|\"','',regex=True)

####Tokenização

Agora, criaremos a nossa lista de tokens. Faremos o mesmo que fizemos com a coluna descricao_local, utilizando o método split().

Para isso, vamos copiar a célula anterior até o str e colar em uma nova célula, adicionando .split() ao final. Dentro dos parênteses, vamos passar como parâmetro o nosso caractere separador, as vírgulas, entre aspas simples ','. Na linha abaixo, escrevemos dados.head() para verificar o resultado:

In [16]:
#Vírgula como separador de token
dados['comodidades'] = dados['comodidades'].str.split(',')
dados.head()

,avaliacao_geral,experiencia_local,max_hospedes,descricao_local,descricao_vizinhanca,quantidade_banheiros,quantidade_quartos,quantidade_camas,modelo_cama,comodidades,taxa_deposito,taxa_limpeza,preco
0,10.0,--,1,"[this, clean, and, comfortable, one, bedroom, ...",Lower Queen Anne is near the Seattle Center (s...,1,1,1,Real Bed,"[Internet, Wireless Internet, Kitchen, Free Pa...",0.0,110.0,0.0
1,10.0,--,1,"[our, century, old, upper, queen, anne, house,...","Upper Queen Anne is a really pleasant, unique ...",1,1,1,Futon,"[TV, Internet, Wireless Internet, Kitchen, Fre...",0.0,45.0,0.0
2,10.0,--,1,"[cozy, room, in, two-bedroom, apartment, along...",The convenience of being in Seattle but on the...,1,1,1,Futon,"[TV, Internet, Wireless Internet, Kitchen, Fre...",0.0,55.0,0.0
3,10.0,--,1,"[very, lovely, and, cozy, room, for, one, conv...","Ballard is lovely, vibrant and one of the most...",1,1,1,Pull-out Sofa,"[Internet, Wireless Internet, Kitchen, Free Pa...",20.0,52.0,0.0
4,10.0,--,1,"[the, studio, at, mibbett, hollow', is, in, a,...",--,1,1,1,Real Bed,"[Wireless Internet, Kitchen, Free Parking on P...",15.0,85.0,0.0


In [17]:
dados['descricao_vizinhanca'][0]

"Lower Queen Anne is near the Seattle Center (space needle, EMP museum, Glass museum, Science Center and Children's museum). It's also near SAM sculpture park, stores, restaurants, SIFF theater and more."

In [18]:
dados['descricao_vizinhanca'] = dados['descricao_vizinhanca'].str.lower()
dados['descricao_vizinhanca'] = dados['descricao_vizinhanca'].str.replace(r'[^a-zA-Z0-9\-\']', ' ', regex=True)
dados['descricao_vizinhanca'] = dados['descricao_vizinhanca'].str.replace(r'(?<!\w)-(?!\w)', '', regex=True)
# Remover stopwords específicas: "e", "ou", "mas"
dados['descricao_vizinhanca'] = dados['descricao_vizinhanca'].str.replace(r'\b(e|ou|mas)\b', '', regex=True)

In [19]:
dados['descricao_vizinhanca'][0]

"lower queen anne is near the seattle center  space needle  emp museum  glass museum  science center and children's museum   it's also near sam sculpture park  stores  restaurants  siff theater and more "

In [20]:
dados['descricao_vizinhanca'] = dados['descricao_vizinhanca'].str.split()
dados['descricao_vizinhanca'].reset_index(inplace=True, drop=True)
dados.head()

,avaliacao_geral,experiencia_local,max_hospedes,descricao_local,descricao_vizinhanca,quantidade_banheiros,quantidade_quartos,quantidade_camas,modelo_cama,comodidades,taxa_deposito,taxa_limpeza,preco
0,10.0,--,1,"[this, clean, and, comfortable, one, bedroom, ...","[lower, queen, anne, is, near, the, seattle, c...",1,1,1,Real Bed,"[Internet, Wireless Internet, Kitchen, Free Pa...",0.0,110.0,0.0
1,10.0,--,1,"[our, century, old, upper, queen, anne, house,...","[upper, queen, anne, is, a, really, pleasant, ...",1,1,1,Futon,"[TV, Internet, Wireless Internet, Kitchen, Fre...",0.0,45.0,0.0
2,10.0,--,1,"[cozy, room, in, two-bedroom, apartment, along...","[the, convenience, of, being, in, seattle, but...",1,1,1,Futon,"[TV, Internet, Wireless Internet, Kitchen, Fre...",0.0,55.0,0.0
3,10.0,--,1,"[very, lovely, and, cozy, room, for, one, conv...","[ballard, is, lovely, vibrant, and, one, of, t...",1,1,1,Pull-out Sofa,"[Internet, Wireless Internet, Kitchen, Free Pa...",20.0,52.0,0.0
4,10.0,--,1,"[the, studio, at, mibbett, hollow', is, in, a,...",[],1,1,1,Real Bed,"[Wireless Internet, Kitchen, Free Parking on P...",15.0,85.0,0.0


In [21]:
dados['descricao_vizinhanca'][0]

['lower',
 'queen',
 'anne',
 'is',
 'near',
 'the',
 'seattle',
 'center',
 'space',
 'needle',
 'emp',
 'museum',
 'glass',
 'museum',
 'science',
 'center',
 'and',
 "children's",
 'museum',
 "it's",
 'also',
 'near',
 'sam',
 'sculpture',
 'park',
 'stores',
 'restaurants',
 'siff',
 'theater',
 'and',
 'more']

#Exercício: trabalhando em outros contextos


Voltemos a pôr em prática tudo o que aprendemos durante as aulas. Disponibilizei novamente os 2 novos conjuntos de dados para download abaixo:

*  Projeto desafio 1: vendas online - dados_vendas_clientes.json;

*  Projeto desafio 2: administração de condomínios - dados_locacao_imoveis.json.

>Lembrando: São dois projetos de tratamento que serão construídos durante o andamento do curso. Portanto, salve seu código de construção em cada desafio para que possa aplicá-lo nos desafios posteriores.

##Etapa 3

*  Projeto desafio 1: vendas online

Na etapa 2, trabalhamos na transformação dos dados numéricos. Agora, podemos trabalhar com valores textuais.

>Devido a uma instabilidade no site da empresa, tivemos problemas com os nomes dos clientes durante o salvamento. Isso resultou em uma coluna de nomes de clientes com uma mistura de letras, maiúsculas e minúsculas, números e outros caracteres.

Sabendo disso, manipule os textos presentes na coluna Cliente para que seja obtido como resultado os nomes dos clientes em letras minúsculas, com a ausência de caracteres especiais ou números.

*  Projeto desafio 2: administração de condomínios

Na etapa 2, trabalhamos na transformação dos dados numéricos. Agora, podemos trabalhar com valores textuais.

>Buscando explicar a organização da identificação dos apartamentos, durante a criação do conjunto de dados, foi adicionado o texto (blocoAP). Esse texto informa que os nomes dos apartamentos estão organizados com a letra do bloco seguida do número do apartamento. No entanto, isso não traz nenhuma informação para nossos dados, sendo interessante realizar a remoção desse texto no conjunto de dados.

Com isso, manipule os textos na coluna apartamento para remover o texto (blocoAP) do DataFrame.


In [22]:
import pandas as pd
import numpy as np
import io

def carregar_dados(url: str, chave: str, coluna_valor: str, prefixo_remover: list[str] = None) -> pd.DataFrame:
    """
    Lê um JSON de uma URL, normaliza os dados, explode listas e limpa a coluna de valores monetários.

    Args:
        url (str): Link do arquivo JSON.
        chave (str): Nome da chave principal do JSON (ex.: 'dados_vendas').
        coluna_valor (str): Nome da coluna que contém os valores monetários.
        prefixo_remover (list[str], opcional): Strings a serem removidas antes de converter em float.

    Returns:
        pd.DataFrame: DataFrame normalizado e com a coluna de valores em float.
    """
    # Carrega e normaliza
    dados = pd.read_json(io.StringIO(pd.read_json(url).to_json()))
    dados = pd.json_normalize(dados[chave])

    # Explode as colunas de listas
    colunas = list(dados.columns)
    dados = dados.explode(colunas[1:]).reset_index(drop=True)

    # Limpeza da coluna de valores
    if prefixo_remover:
        for item in prefixo_remover:
            dados[coluna_valor] = dados[coluna_valor].str.replace(item, '', regex=False)

    dados[coluna_valor] = dados[coluna_valor].str.replace(',', '.', regex=False).str.strip()
    dados[coluna_valor] = dados[coluna_valor].astype(np.float64)

    return dados


url_vendas = "https://raw.githubusercontent.com/YuriArduino/Estudos_Pandas/refs/heads/data-tests/dados_vendas_clientes.json"
url_locacao = "https://raw.githubusercontent.com/YuriArduino/Estudos_Pandas/refs/heads/data-tests/dados_locacao_imoveis.json"

dados_vendas = carregar_dados(url_vendas, chave="dados_vendas", coluna_valor="Valor da compra", prefixo_remover=["R$ "])
dados_locacao = carregar_dados(url_locacao, chave="dados_locacao", coluna_valor="valor_aluguel", prefixo_remover=["$", " reais"])

print(dados_vendas.head())
print(dados_locacao.head())

  Data de venda             Cliente  Valor da compra
0    06/06/2022     @ANA _LUCIA 321           836.50
1    06/06/2022  DieGO ARMANDIU 210           573.33
2    06/06/2022  DieGO ARMANDIU 210           392.80
3    06/06/2022  DieGO ARMANDIU 210           512.34
4    07/06/2022  Isabely JOanes 738           825.31
      apartamento datas_combinadas_pagamento datas_de_pagamento  valor_aluguel
0  A101 (blocoAP)                 01/06/2022         05/06/2022         1000.0
1  A101 (blocoAP)                 01/07/2022         03/07/2022         2500.0
2  A102 (blocoAP)                 02/06/2022         02/06/2022         1100.0
3  A102 (blocoAP)                 02/07/2022         06/07/2022         2600.0
4  B201 (blocoAP)                 03/06/2022         07/06/2022         1200.0


### Projeto 1

In [23]:
dados_vendas.head()

,Data de venda,Cliente,Valor da compra
0,06/06/2022,@ANA _LUCIA 321,836.50
1,06/06/2022,DieGO ARMANDIU 210,573.33
2,06/06/2022,DieGO ARMANDIU 210,392.80
3,06/06/2022,DieGO ARMANDIU 210,512.34
4,07/06/2022,Isabely JOanes 738,825.31


In [24]:
dados_vendas['Cliente'] = dados_vendas['Cliente'].str.lower()
dados_vendas['Cliente'] = dados_vendas['Cliente'].str.replace(r'[^a-z]', ' ', regex=True)

In [25]:
dados_vendas['Cliente'].head()

,Cliente
0,ana lucia
1,diego armandiu
2,diego armandiu
3,diego armandiu
4,isabely joanes


### Projeto 2

In [26]:
dados_locacao.head()

,apartamento,datas_combinadas_pagamento,datas_de_pagamento,valor_aluguel
0,A101 (blocoAP),01/06/2022,05/06/2022,1000.0
1,A101 (blocoAP),01/07/2022,03/07/2022,2500.0
2,A102 (blocoAP),02/06/2022,02/06/2022,1100.0
3,A102 (blocoAP),02/07/2022,06/07/2022,2600.0
4,B201 (blocoAP),03/06/2022,07/06/2022,1200.0


In [29]:
dados_locacao['apartamento'] = dados_locacao['apartamento'].str.replace('(blocoAP)', '', regex=False)
dados_locacao.head()

,apartamento,datas_combinadas_pagamento,datas_de_pagamento,valor_aluguel
0,A101,01/06/2022,05/06/2022,1000.0
1,A101,01/07/2022,03/07/2022,2500.0
2,A102,02/06/2022,02/06/2022,1100.0
3,A102,02/07/2022,06/07/2022,2600.0
4,B201,03/06/2022,07/06/2022,1200.0


---------------------

# 🔍 Guia Completo de Regex: Operador ^ e Exemplos Práticos

## 📖 String de Exemplo

Vamos usar a string `abc123!@#` para demonstrar todos os exemplos práticos de regex e entender claramente como o operador `^` funciona dentro e fora dos colchetes:

```text
abc123!@#
```

---

## 🎯 Exemplos Práticos de Regex

### 1. `[a-z]` - Letras Minúsculas

**Descrição:** Faz **match apenas com letras minúsculas de a-z**

**Resultado em `abc123!@#`:**
```
✅ Matches encontrados: a, b, c
❌ Não faz match com: 1, 2, 3, !, @, #
```

**Uso prático:**
```javascript
"abc123!@#".match(/[a-z]/g) // ['a', 'b', 'c']
```

---

### 2. `[^a-z]` - Negação de Letras Minúsculas

**Descrição:** Faz **match com tudo que NÃO for letra minúscula** (o `^` dentro dos colchetes significa negação)

**Resultado em `abc123!@#`:**
```
❌ Não faz match com: a, b, c
✅ Matches encontrados: 1, 2, 3, !, @, #
```

**Uso prático:**
```javascript
// Remove tudo que não for letra minúscula
"abc123!@#".replace(/[^a-z]/g, '') // "abc"
```

---

### 3. `[0-9]` - Números

**Descrição:** Casa **apenas dígitos de 0-9**

**Resultado em `abc123!@#`:**
```
❌ Não casa com: a, b, c, !, @, #
✅ Matches encontrados: 1, 2, 3
```

**Uso prático:**
```javascript
"abc123!@#".match(/[0-9]/g) // ['1', '2', '3']
```

---

### 4. `[^0-9]` - Negação de Números

**Descrição:** Casa **tudo que NÃO for dígito**

**Resultado em `abc123!@#`:**
```
✅ Matches encontrados: a, b, c, !, @, #
❌ Não casa com: 1, 2, 3
```

**Uso prático:**
```javascript
// Remove tudo que não for número
"abc123!@#".replace(/[^0-9]/g, '') // "123"
```

---

### 5. `^a` - Início da String

**Descrição:** O `^` **fora dos colchetes** significa **início da string**. Casa apenas se a string começar com 'a'

**Resultado em `abc123!@#`:**
```
✅ Match encontrado: a (primeira posição)
```

**Uso prático:**
```javascript
/^a/.test("abc123!@#") // true
/^b/.test("abc123!@#") // false
```

---

### 6. `#$` - Final da String

**Descrição:** O `$` significa **final da string**. Casa apenas se a string terminar com '#'

**Resultado em `abc123!@#`:**
```
✅ Match encontrado: # (última posição)
```

**Uso prático:**
```javascript
/#$/.test("abc123!@#") // true
/!$/.test("abc123!@#") // false
```

---

## 🧠 Diferenças Importantes do Operador `^`

### Dentro dos Colchetes `[^...]`
- **Função:** Negação/Exclusão
- **Significa:** "Qualquer coisa EXCETO os caracteres listados"
- **Exemplo:** `[^abc]` = qualquer caractere que não seja a, b ou c

### Fora dos Colchetes `^...`
- **Função:** Âncora de início
- **Significa:** "A string deve COMEÇAR com..."
- **Exemplo:** `^abc` = a string deve começar com "abc"

---

## 🛠️ Casos de Uso Comuns

### Limpeza de Dados

```javascript
// Manter apenas letras
texto.replace(/[^a-zA-Z]/g, '')

// Manter apenas números
texto.replace(/[^0-9]/g, '')

// Manter apenas letras, números e espaços
texto.replace(/[^a-zA-Z0-9 ]/g, '')
```

### Validações

```javascript
// Verificar se começa com letra maiúscula
/^[A-Z]/.test(nome)

// Verificar se termina com ponto
/\.$/.test(frase)

// Verificar se contém apenas números
/^[0-9]+$/.test(codigo)
```

---

## 📝 Resumo Visual

| Regex | Significado | Resultado em `abc123!@#` |
|-------|-------------|---------------------------|
| `[a-z]` | Apenas letras minúsculas | `a`, `b`, `c` |
| `[^a-z]` | Tudo exceto letras minúsculas | `1`, `2`, `3`, `!`, `@`, `#` |
| `[0-9]` | Apenas números | `1`, `2`, `3` |
| `[^0-9]` | Tudo exceto números | `a`, `b`, `c`, `!`, `@`, `#` |
| `^a` | Começa com 'a' | `a` (início) |
| `#$` | Termina com '#' | `#` (final) |

---

## 🎓 Dica Final

**Lembre-se sempre:**
- **`^` dentro de `[]`** = negação ("não quero isso")
- **`^` fora de `[]`** = início da string ("deve começar assim")
- **`$`** = final da string ("deve terminar assim")

Essa distinção é fundamental para dominar regex e evitar confusões!